In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import osmium
import sqlite3
import os

rng = np.random.default_rng(42)
CRS = "EPSG:31370"  # Belgian Lambert 72

### Census 2021 total counts from home muni to BCR

In [ ]:
# Get number of people working in Brussels per home municipality from the 2021 census
expected_number_of_commuters = 416208

df_raw = pd.read_excel(
    'input_data/working_population_home_work_gender.xlsx',
    sheet_name=6,
    header=None
)

# Column 6 = Brussels workplace column
# Column 0 = NIS code, Column 1 = municipality name, Column 2 = sex (Hommes/Femmes/Total)

# Extract relevant columns
df_parsed = df_raw[[0, 1, 2, 6]].copy()
df_parsed.columns = ['nis_code', 'municipality', 'sex', 'works_in_brussels']

# Forward fill NIS code and municipality name
df_parsed['nis_code'] = df_parsed['nis_code'].ffill()
df_parsed['municipality'] = df_parsed['municipality'].ffill()

# Drop header rows where sex is not Hommes/Femmes/Total
df_parsed = df_parsed[df_parsed['sex'].isin(['Hommes', 'Femmes', 'Total'])].copy()

df_parsed['nis_code'] = pd.to_numeric(df_parsed['nis_code'], errors='coerce')
df_parsed['works_in_brussels'] = pd.to_numeric(df_parsed['works_in_brussels'], errors='coerce')
df_parsed = df_parsed.dropna(subset=['nis_code'])
df_parsed['nis_code'] = df_parsed['nis_code'].astype(int)

# Keep only Total rows
df_total = df_parsed[df_parsed['sex'] == 'Total'].copy()

# Exclude Brussels municipalities and aggregate rows
df_commuters_census = df_total[
    (df_total['nis_code'] >= 10000) &
    (~df_total['nis_code'].between(21000, 21999))
].copy()

# Exclude provinces and arrondissements
exclude_codes = [20001, 20002]  # Brabant flamand and wallon provinces

df_municipalities = df_commuters_census[
    (df_commuters_census['nis_code'] >= 10000) &
    (df_commuters_census['nis_code'] % 1000 != 0) &  # exclude provinces and arrondissements
    (~df_commuters_census['nis_code'].between(21000, 21999)) & # exclude Brussels
    (~df_commuters_census['nis_code'].isin(exclude_codes))
].copy()

print(f"Total municipalities: {len(df_municipalities)}")
print(f"Total commuters to Brussels: {df_municipalities['works_in_brussels'].sum():.0f}")

if df_municipalities['works_in_brussels'].sum() != expected_number_of_commuters:
    print(f"Warning: Total commuters from census ({df_municipalities['works_in_brussels'].sum():.0f}) does not match expected number ({expected_number_of_commuters}).")

### Process HTS data

In [ ]:
# Load HTS data and spatial data
df_commuters = pd.read_csv('input_data/hts_outside_brussels.csv')
df_company_car = pd.read_csv('input_data/hts_commuters_company_car.csv') # we only started to tack company car later, so join this separately
df_weights = pd.read_excel('input_data/weights_individuals.xlsx')
gdf_sectors = gpd.read_file('input_data/statistical_sectors.shp')
df_postal = pd.read_excel('input_data/postal_code_refnis_code.xlsx')

# Merge weights
df_commuters = df_commuters.merge(df_weights, left_on='participant_id', right_on='p_id', how='left')
df_commuters = df_commuters.merge(df_company_car[['participant_id', 'has_company_car']], on='participant_id', how='left')

In [ ]:
# Match HTS commuters to NIS codes via postal code
df_hts = df_commuters.merge(
    df_postal.rename(columns={'Postal code': 'postcode', 'Refnis code': 'nis_code'}),
    left_on='postcode_home',
    right_on='postcode',
    how='left'
)

print(f"\nHTS commuters matched to NIS code: {df_hts['nis_code'].notna().sum()}")
print(f"HTS commuters unmatched: {df_hts['nis_code'].isna().sum()}")

# Check how many unique municipalities are represented in HTS
print(f"\nUnique municipalities in HTS: {df_hts['nis_code'].nunique()}")
print(f"Unique municipalities in Census: {len(df_municipalities)}")

# Check coverage - what % of Census commuters are in municipalities represented in HTS
hts_munis = df_hts['nis_code'].dropna().unique()
census_coverage = df_municipalities[
    df_municipalities['nis_code'].isin(hts_munis)
]['works_in_brussels'].sum()
print(f"\nCensus commuters in HTS-represented municipalities: {census_coverage:.0f}")
print(f"Coverage: {census_coverage/416208*100:.1f}%")

In [ ]:
# Add province code to both datasets
# Province code = first 2 digits of NIS code (for 5-digit codes)
df_municipalities['province_code'] = (df_municipalities['nis_code'] // 1000).astype(int)
df_hts['province_code'] = (df_hts['nis_code'] // 1000).astype(int)

# Build HTS lookup by municipality and province
hts_by_municipality = {}
for nis, group in df_hts.groupby('nis_code'):
    hts_by_municipality[nis] = group

hts_by_province = {}
for prov, group in df_hts.groupby('province_code'):
    hts_by_province[prov] = group

# Brussels-wide fallback
hts_all = df_hts.copy()

# Resample commuters per municipality
all_commuters = []

for _, row in df_municipalities.iterrows():
    nis = row['nis_code']
    target_n = int(row['works_in_brussels'])
    province = row['province_code']
    
    if target_n == 0:
        continue
    
    # Get HTS pool for this municipality
    if nis in hts_by_municipality:
        if len(hts_by_municipality[nis]) >= 3:
            pool = hts_by_municipality[nis]
            source = 'municipality'
        else:
            pool = hts_by_province[province]
            source = 'province'
    elif province in hts_by_province:
        pool = hts_by_province[province]
        source = 'province'
    else:
        pool = hts_all
        source = 'national'
    
    # Weighted resampling
    weights = pool['WeightPOP'].values
    weights = weights / weights.sum()
    
    sampled = pool.sample(
        n=target_n,
        replace=True,
        weights=weights,
        random_state=rng.integers(1e6)
    )
    sampled = sampled.copy()
    sampled['municipality_nis'] = nis
    sampled['home_municipality'] = row['municipality']
    sampled['resample_source'] = source
    all_commuters.append(sampled)

df_commuters_new = pd.concat(all_commuters, ignore_index=True)

print(f"Total commuters generated: {len(df_commuters_new)}")
print("\nResample source distribution:")
print(df_commuters_new['resample_source'].value_counts())
print("\nAge distribution:")
print(df_commuters_new['age_group'].value_counts().sort_index())
print("\nGender distribution:")
print(df_commuters_new['sex'].value_counts(normalize=True) * 100)


# Assign home address

In [ ]:
# Get all nodes on highways in the OSM data
class WayNodeHandler(osmium.SimpleHandler):
    def __init__(self):
        super().__init__()
        self.highway_node_ids = set()
    
    def way(self, w):
        if 'highway' in w.tags:
            for n in w.nodes:
                self.highway_node_ids.add(n.ref)

class NodeLocationHandler(osmium.SimpleHandler):
    def __init__(self, node_ids):
        super().__init__()
        self.node_ids = node_ids
        self.nodes = []
    
    def node(self, n):
        if n.id in self.node_ids:
            self.nodes.append({
                'node_id': n.id,
                'lat': n.location.lat,
                'lon': n.location.lon
            })

# Step 1: collect node IDs from highway ways
way_handler = WayNodeHandler()
way_handler.apply_file('input_data/belgium.osm.pbf')
print(f"Highway node IDs collected: {len(way_handler.highway_node_ids)}")

# Step 2: extract coordinates for those nodes
node_handler = NodeLocationHandler(way_handler.highway_node_ids)
node_handler.apply_file('input_data/belgium.osm.pbf', locations=True)

df_nodes = pd.DataFrame(node_handler.nodes)
print(f"Total highway nodes with coordinates: {len(df_nodes)}")

In [ ]:
# Convert to GeoDataFrame in WGS84 then reproject to Lambert
gdf_nodes = gpd.GeoDataFrame(
    df_nodes,
    geometry=gpd.points_from_xy(df_nodes['lon'], df_nodes['lat']),
    crs='EPSG:4326'
).to_crs(CRS)

gdf_nodes['x'] = gdf_nodes.geometry.x
gdf_nodes['y'] = gdf_nodes.geometry.y

# Load municipality boundaries from statistical sectors shapefile
gdf_sectors = gpd.read_file('input_data/statistical_sectors.shp').to_crs(CRS)
gdf_sectors['CNIS5_2020'] = gdf_sectors['CNIS5_2020'].astype(str).str.zfill(5)

# Dissolve sectors to municipality level
gdf_municipalities = gdf_sectors.dissolve(by='CNIS5_2020').reset_index()
print(f"Total municipalities: {len(gdf_municipalities)}")

# Spatial join nodes to municipalities
gdf_nodes_joined = gpd.sjoin(
    gdf_nodes,
    gdf_municipalities[['CNIS5_2020', 'geometry']],
    how='inner',
    predicate='within'
)

print(f"Nodes with municipality assignment: {len(gdf_nodes_joined)}")
print(f"Municipalities with at least one node: {gdf_nodes_joined['CNIS5_2020'].nunique()}")

# Build lookup dictionary
nodes_by_municipality = gdf_nodes_joined.groupby('CNIS5_2020').apply(
    lambda x: x[['x', 'y']].values.tolist()
).to_dict()

print("\nSample municipality node counts:")
for muni, nodes in list(nodes_by_municipality.items())[:5]:
    print(f"  {muni}: {len(nodes)} nodes")

In [ ]:
# Reassign commuter home coordinates to OSM nodes
df_commuters = df_commuters_new.copy()

# Normalize commuter municipality codes to 5-char strings
df_commuters['municipality_nis'] = (
    pd.to_numeric(df_commuters['municipality_nis'], errors='coerce')
      .astype('Int64')
      .astype('string')
      .str.zfill(5)
)

# Check which commuter municipalities have no nodes
missing_munis = set(df_commuters['municipality_nis'].unique()) - set(nodes_by_municipality.keys())
print(f"Commuter municipalities with no nodes: {len(missing_munis)}")
print(f"Missing municipalities: {missing_munis}")
print(f"Affected commuters: {df_commuters['municipality_nis'].isin(missing_munis).sum()}")

# For missing municipalities, find nearest municipality with nodes
# Build centroid lookup for fallback
muni_centroids = gdf_municipalities.copy()
muni_centroids['centroid_x'] = muni_centroids.geometry.centroid.x
muni_centroids['centroid_y'] = muni_centroids.geometry.centroid.y
muni_centroids = muni_centroids[['CNIS5_2020', 'centroid_x', 'centroid_y']].set_index('CNIS5_2020')

def get_nearest_muni_with_nodes(muni_nis):
    """Find nearest municipality that has nodes."""
    if muni_nis not in muni_centroids.index:
        return None
    cx = muni_centroids.loc[muni_nis, 'centroid_x']
    cy = muni_centroids.loc[muni_nis, 'centroid_y']
    
    # Check all municipalities with nodes
    best_muni = None
    best_dist = float('inf')
    for candidate_muni in nodes_by_municipality.keys():
        if candidate_muni not in muni_centroids.index:
            continue
        dx = muni_centroids.loc[candidate_muni, 'centroid_x'] - cx
        dy = muni_centroids.loc[candidate_muni, 'centroid_y'] - cy
        dist = np.sqrt(dx**2 + dy**2)
        if dist < best_dist:
            best_dist = dist
            best_muni = candidate_muni
    return best_muni

# Build fallback mapping for missing municipalities
print("Building fallback mapping for missing municipalities...")
fallback_mapping = {}
for muni in missing_munis:
    fallback_mapping[muni] = get_nearest_muni_with_nodes(muni)

# Assign random OSM node as home location
def assign_home_node(municipality_nis, rng):
    """Pick a random OSM node from the agent's municipality."""
    muni = municipality_nis
    if muni not in nodes_by_municipality:
        muni = fallback_mapping.get(muni)
    if muni is None:
        return None, None
    nodes = nodes_by_municipality[muni]
    chosen = nodes[rng.integers(len(nodes))]
    return chosen[0], chosen[1]  # x, y

rng = np.random.default_rng(42)

print("Assigning home nodes to commuters...")
results = df_commuters['municipality_nis'].apply(
    lambda m: pd.Series(assign_home_node(m, rng), index=['home_x', 'home_y'])
)

df_commuters['home_x'] = results['home_x']
df_commuters['home_y'] = results['home_y']

print(f"Missing home coordinates: {df_commuters['home_x'].isna().sum()}")
print(f"Sample coordinates:")
print(df_commuters[['municipality_nis', 'home_x', 'home_y']].head())

# Get commuting probabilities

In [ ]:
def calculate_commute_probabilities(db_path):
    """
    Calculate the probability that a person from statistical sector i commutes to sector j.

    Args:
        db_path: Path to the SQLite database file

    Returns:
        DataFrame with home_sector, work_sector, and probability columns
    """
    # Connect to the database
    conn = sqlite3.connect(db_path)
    # Check if the database file exists

    if not os.path.exists(db_path):
        raise FileNotFoundError(f"The database file at {db_path} does not exist.")

    # Read the data
    query = """
    SELECT 
        CD_SECTOR_RESIDENCE AS home_statistical_sector,
        CD_SECTOR_WORK AS work_statistical_sector, 
        CAST(OBS_VALUE AS INTEGER) AS num_people
    FROM 'VU_GEO_LPW_MATRIX'
    """
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Extract the part after underscore for sector codes
    df["home_statistical_sector"] = df["home_statistical_sector"].str.split("_").str[1]

    # Calculate total people from each home sector
    total_by_home = (
        df.groupby("home_statistical_sector")["num_people"].sum().reset_index()
    )
    total_by_home.columns = ["home_statistical_sector", "total_home"]

    # Merge and calculate probabilities
    df = df.merge(total_by_home, on="home_statistical_sector")
    df["probability"] = df["num_people"] / df["total_home"]

    # Select final columns
    result = df[["home_statistical_sector", "work_statistical_sector", "probability", "num_people"]]

    return result


In [ ]:
db_path = "input_data/census_matrix_home_work_ss_2011.sqlite"

probabilities = calculate_commute_probabilities(db_path)
print(probabilities.head())

# Save to CSV, already done
probabilities.to_csv("output/commute_probabilities_all.csv", index=False)

# Assign work sector

In [ ]:
probabilities = pd.read_csv(
    "output/commute_probabilities_all.csv"
)

probabilities['work_statistical_sector'] = probabilities['work_statistical_sector'].str.split('_').str[-1]

# Keep only sectors with an actual Belgian statistical sector code
mask_valid_work_sector = (
    ~probabilities["work_statistical_sector"].str.contains("ZZZZ", na=False)
    & probabilities["work_statistical_sector"].ne("FOR")
)

# MUNICIPALITY-LEVEL PROBABILITIES
muni_probabilities = probabilities[mask_valid_work_sector].copy()

# Add municipality from work sector (first 11 characters)
muni_probabilities["work_municipality"] = muni_probabilities["work_statistical_sector"].str[:11]

# Probability of each work sector within (home sector, work municipality)
muni_probabilities["probability"] = (
    muni_probabilities["num_people"]
    / muni_probabilities.groupby(["home_statistical_sector", "work_municipality"])["num_people"].transform("sum")
)

od_prob_municipality = {}
for index, group in muni_probabilities.groupby(["home_statistical_sector", "work_municipality"]):
    od_prob_municipality[index] = {
        "work_sectors": group["work_statistical_sector"].values,
        "probs": group["probability"].values,
    }

# Fallback distribution by municipality only (ignores home sector)
od_prob_by_muni = {}
for municipality_id, group in muni_probabilities.groupby("work_municipality"):
    probs = group["num_people"] / group["num_people"].sum()
    od_prob_by_muni[municipality_id] = {
        "work_sectors": group["work_statistical_sector"].values,
        "probs": probs.values,
    }

In [ ]:
# Assign work sector to commuters

# Brussels municipality codes (21 communes, NIS codes start with 21)
brussels_sectors = muni_probabilities[
    muni_probabilities['work_statistical_sector'].str.startswith('21')
]['work_statistical_sector'].unique()

# Filter OD matrix to Brussels work sectors only
brussels_od = muni_probabilities[
    muni_probabilities['work_statistical_sector'].str.startswith('21')
].copy()

# Actually for commuters: home is outside Brussels, work is in Brussels
# So we aggregate: for each home_municipality (outside Brussels), 
# what is the probability of each Brussels work sector?
brussels_od_commuters = probabilities[
    mask_valid_work_sector
    & probabilities['work_statistical_sector'].str.startswith('21')
].copy()

brussels_od_commuters['home_municipality'] = brussels_od_commuters['home_statistical_sector'].str[:11]

od_commuter_by_home_muni = {}
for home_muni, group in brussels_od_commuters.groupby('home_municipality'):
    total = group['num_people'].sum()
    od_commuter_by_home_muni[home_muni] = {
        'work_sectors': group['work_statistical_sector'].values,
        'probs': (group['num_people'] / total).values
    }

# Fallback: overall Brussels work sector distribution (ignores home municipality)
brussels_overall = brussels_od_commuters.groupby('work_statistical_sector')['num_people'].sum().reset_index()
brussels_fallback = {
    'work_sectors': brussels_overall['work_statistical_sector'].values,
    'probs': (brussels_overall['num_people'] / brussels_overall['num_people'].sum()).values
}

def assign_work_sector_commuter(home_municipality_nis, rng):
    """Assign Brussels work sector based on home municipality."""
    lookup = od_commuter_by_home_muni.get(home_municipality_nis)
    if lookup is None:
        # Fallback to overall Brussels distribution
        lookup = brussels_fallback
    return rng.choice(lookup['work_sectors'], p=lookup['probs'])

df_commuters['work_sector'] = df_commuters['municipality_nis'].apply(
    lambda m: assign_work_sector_commuter(m, rng)
)

print(f"Missing work sectors: {df_commuters['work_sector'].isna().sum()}")
print(f"Sample work sectors:\n{df_commuters['work_sector'].value_counts().head(10)}")

In [ ]:
df_commuters.to_csv("output/commuters_with_work_sectors_new.csv", index=False)

# Assign work address

In [ ]:
def get_work_addresses():
    """
    Load addresses and filter for those suitable for office/work use.
    For Brussels sectors: use actual addresses from work zones (land use).
    For non-Brussels sectors: use the centroid of the statistical sector.

    Returns:
        Dictionary mapping sector to list of work addresses (geometries)
    """
    # Load addresses file
    addresses = gpd.read_file("input_data/urbis_addresses.shp")

    # Load land use/parcel descriptions
    land_use = gpd.read_file("input_data/pras.shp")

    # Filter for office/work land use types
    work_land_use = land_use[
        land_use["AFFECTATIO"].isin(
            [
                "AfZFMixite",
                "AfZIndustrie",
                "AfZTransPort",
                "AfZAdmin",
                "AfZEquipt",
                "AfZMixite",
                "AfZEntrMilUrb",
                "AfZChFer",
                "AfZir",
            ]
        )
    ]

    # Ensure both use the same CRS format (EPSG:31370 = Belgian Lambert 72)
    addresses = addresses.to_crs(CRS)
    work_land_use = work_land_use.to_crs(CRS)

    # Merge addresses with land use data and filter for work zones
    sector_addresses = {}
    addresses_with_landuse = addresses.sjoin(work_land_use, how="inner")

    for sector, group in addresses_with_landuse.groupby("STATNISCOD"):
        sector_addresses[sector] = group.geometry.tolist()

    # Load statistical sectors for all sectors (including non-Brussels)
    statistical_sectors = gpd.read_file(
        "input_data/statistical_sectors.shp"
    )
    statistical_sectors = statistical_sectors.to_crs(CRS)

    # For sectors without work addresses, add their centroid
    for sector in statistical_sectors["CS01012020"].unique():
        if sector not in sector_addresses:
            # Get the centroid of this statistical sector
            sector_geom = statistical_sectors[
                statistical_sectors["CS01012020"] == sector
            ]
            if not sector_geom.empty:
                centroid = sector_geom.geometry.centroid.values[0]
                sector_addresses[sector] = [centroid]

    return sector_addresses

In [ ]:
# Get all possible work addresses (offices/industrial zones etc) for Brussels
# + centroids of statistical sectors outside of Brussels
work_addresses = get_work_addresses()

In [ ]:
def assign_work_address(synthetic_pop, sector_addresses):
    """
    Assign a random work address to each agent based on their assigned work sector.

    Args:
        synthetic_pop: DataFrame with synthetic population (must include 'work_sector' column)
        sector_addresses: Dictionary mapping work_sector to a list of possible addresses (geometries)

    Returns:
        DataFrame with an additional 'work_geometry' column assigned to each agent
    """

    def assign_random_work_address(work_sector):
        work_sector = normalize_sector(work_sector)
        if work_sector in sector_addresses:
            return np.random.choice(sector_addresses[work_sector])
        else:
            return None  # If no addresses for this sector

    # Handle sector matching: extract part after underscore or keep 'FOR'
    def normalize_sector(sector):
        if pd.isna(sector):
            return None
        if sector == "FOR":
            return "FOR"
        # Extract part after underscore if present
        if "_" in sector:
            return sector.split("_", 1)[1]
        return sector
    
    synthetic_pop["work_geometry"] = synthetic_pop["assigned_work_sector"].apply(
        assign_random_work_address
    )

    # Convert to GeoDataFrame
    pop_gdf = gpd.GeoDataFrame(synthetic_pop, geometry="work_geometry", crs=CRS)

    # Extract X/Y coordinates if needed
    pop_gdf["work_x"] = pop_gdf.geometry.x
    pop_gdf["work_y"] = pop_gdf.geometry.y

    synthetic_pop["work_x"] = pop_gdf["work_x"]
    synthetic_pop["work_y"] = pop_gdf["work_y"]

    # Save to file
    pop_gdf.to_file("output/workers_with_work_addresses.shp")
    pop_gdf[
        [
            "participant_id",
            "day_of_trip",
            "sex",
            "age_group_broad",
            "age_in_2016",
            "education_detail",
            "education",
            "profession_code",
            "profession",
            "work_type",
            "has_car",
            "has_company_car",
            "income_category",
            "postcode_home",
            "postcode_work",
            "lives_in_brussels",
            "works_in_brusels",
            "region",
            "age_group",
            "departure_home_to_work",
            "arrival_work",
            "trip_duration",
            "departure_from_work",
            "distance_km",
            "mode",
            "municipality_nis",
            "home_x",
            "home_y",
            "work_sector",
            "work_x",
            "work_y",
        ]
    ].to_csv("output/commuters_with_addresses_new.csv", index=False)

    return synthetic_pop

In [ ]:
# Rename to match what assign_work_address expects
df_commuters['assigned_work_sector'] = df_commuters['work_sector']

# Reuse existing work_addresses lookup (already loaded)
df_commuters = assign_work_address(df_commuters, work_addresses)

In [ ]:
# Compute approximate distance home-work (km)
DETOUR_FACTOR = 1.3

df_commuters["commute_dist_km"] = (
    np.sqrt(
        (df_commuters["home_x"] - df_commuters["work_x"])**2 +
        (df_commuters["home_y"] - df_commuters["work_y"])**2
    ) / 1000 * DETOUR_FACTOR
)

In [ ]:
# Remove agents who travel on weekends
n_before = len(df_commuters)

df_commuters = df_commuters.loc[
    ~df_commuters["day_of_trip"].isin([1, 7])
].copy()

n_after = len(df_commuters)
n_removed = n_before - n_after

print(f"Removed weekend agents: {n_removed} / {n_before} ({100 * n_removed / n_before:.2f}%)")
print(f"Remaining agents: {n_after}")

In [ ]:
# Impute missing departure_from_work
hts = pd.read_csv('input_data/hts_outside_brussels.csv')
weights = pd.read_excel("input_data/weights_individuals.xlsx")
hts = hts.merge(weights, left_on='participant_id', right_on='p_id', how='left')
hts = hts.rename(columns={"WeightPOP": "weight"})

DIST_BINS   = [0, 5, 10, 15, 30, np.inf]
DIST_LABELS = ["0-5", "5-10", "10-15", "15-30", "30+"]

# Distance bracket for HTS (source for duration sampling)
hts["dist_bracket"] = pd.cut(
    hts["distance_km"], bins=DIST_BINS, labels=DIST_LABELS, right=False
)

# Distance bracket for synthetic commuters (target to impute)
df_commuters["dist_bracket"] = pd.cut(
    pd.to_numeric(df_commuters["commute_dist_km"], errors="coerce"),
    bins=DIST_BINS,
    labels=DIST_LABELS,
    right=False
)

# Ensure work_duration exists for all rows
if "work_duration" not in df_commuters.columns:
    df_commuters["work_duration"] = df_commuters["departure_from_work"] - df_commuters["arrival_work"]

hts["work_duration"] = hts["departure_from_work"] - hts["arrival_work"]

# Build duration distribution per distance bracket from HTS 
# (use all records that have a valid duration)
duration_pools = {}
for bracket, grp in hts.groupby("dist_bracket", observed=True):
    valid = grp[grp["work_duration"].notna()][["work_duration", "weight"]]
    if len(valid) >= 3:
        duration_pools[bracket] = valid.reset_index(drop=True)

# Fallback: overall duration pool ignoring bracket
duration_fallback = hts[hts["work_duration"].notna()][["work_duration", "weight"]].reset_index(drop=True)

def sample_duration(bracket, rng):
    pool = duration_pools.get(bracket, duration_fallback)
    w = pool["weight"].values.astype(float)
    w = w / w.sum()
    idx = rng.choice(len(pool), p=w)
    return pool.iloc[idx]["work_duration"]

# Apply to all agents — use reported value where available, impute where not
mask_missing_return = df_commuters["departure_from_work"].isna()
print(f"Agents missing departure_from_work: {mask_missing_return.sum()}")

for idx in df_commuters[mask_missing_return].index:
    bracket  = df_commuters.loc[idx, "dist_bracket"]
    arrival  = df_commuters.loc[idx, "arrival_work"]
    duration = sample_duration(bracket, rng)
    df_commuters.loc[idx, "departure_from_work"] = arrival + duration

# Recompute work duration and correct overnight wrap-around
raw_duration = df_commuters["departure_from_work"] - df_commuters["arrival_work"]
overnight_mask = raw_duration < 0
df_commuters["work_duration"] = raw_duration.where(raw_duration >= 0, raw_duration + 1440)

print(f"Overnight shifts corrected (+24h): {overnight_mask.sum()}")
print(f"Negative work_duration after correction: {(df_commuters['work_duration'] < 0).sum()}")

# Cap departures that push past midnight (1440 min)
over_midnight = df_commuters["departure_from_work"] > 1440
print(f"Departures past midnight (capping): {over_midnight.sum()}")
df_commuters.loc[over_midnight, "departure_from_work"] = 1440

print("\n══ Final departure_from_work distribution ══")
fmt = lambda m: f"{int(m)//60:02d}:{int(m)%60:02d}"
desc = df_commuters["departure_from_work"].describe()
for stat, val in desc.items():
    print(f"  {stat}: {fmt(val) if stat not in ['count','std'] else round(val)}")

print("\n══ Work duration after imputation ══")
print(df_commuters["work_duration"].describe().round(0))

# Remove teleworking

In [ ]:
# Tag teleworking commuters for today (do not remove yet)
# These numbers are based on the Teleworking survey 2018, adjusted for the increase after COVID-19.
# https://mobilit.belgium.be/sites/default/files/documents/publications/2023/Chiffres-cl%C3%A9s-du-t%C3%A9l%C3%A9travail.pdf
# https://mobilit.belgium.be/fr/mobilite-durable/enquetes-et-resultats/teletravail-et-mobilite

TARGET_TELEWORK_SHARE = 0.16      # 16% of commuters telework today
TARGET_TELEWORK_KM_SHARE = 0.15   # aim for ~15% of commute km

n_agents = len(df_commuters)
n_telework = int(round(TARGET_TELEWORK_SHARE * n_agents))

# Base propensity by education (Higher educated ≈ 2x Lower educated in relative odds)
education_weight = np.where(
    df_commuters["education"].eq("Higher educated"),
    1.744,
    1.0
).astype(float)

# Distance effect: longer commute => higher telework chance
dist = pd.to_numeric(df_commuters["commute_dist_km"], errors="coerce")
dist = dist.fillna(dist.median())
dist_values = dist.to_numpy()

# Convert distance to rank percentile in [0, 1]
dist_rank = dist.rank(method="average", pct=True).to_numpy()

# Use separate RNG for telework assignment (reproducible, isolated from other draws)
rng_telework = np.random.default_rng(42)

def draw_teleworkers(distance_gamma, seed):
    """Draw exactly n_telework commuters for a given distance exponent."""
    distance_weight = (1.0 + dist_rank) ** distance_gamma
    telework_score = education_weight * distance_weight
    telework_prob = telework_score / telework_score.sum()

    local_rng = np.random.default_rng(seed)
    tele_idx = local_rng.choice(
        df_commuters.index.to_numpy(),
        size=n_telework,
        replace=False,
        p=telework_prob
    )

    km_total_local = np.nansum(dist_values)
    km_tele_local = np.nansum(df_commuters.loc[tele_idx, "commute_dist_km"].to_numpy())
    km_share_local = km_tele_local / km_total_local if km_total_local > 0 else np.nan
    return tele_idx, km_share_local

# Calibrate distance exponent so expected km share is close to target
low_gamma, high_gamma = 0.2, 8.0
best_gamma = 1.0
best_gap = float("inf")

for _ in range(16):
    gamma = (low_gamma + high_gamma) / 2
    km_shares = []
    # Average over a few seeds for a stable estimate
    for s in range(5):
        _, km_share_tmp = draw_teleworkers(gamma, seed=1000 + s)
        km_shares.append(km_share_tmp)
    mean_km_share = float(np.nanmean(km_shares))

    gap = abs(mean_km_share - TARGET_TELEWORK_KM_SHARE)
    if gap < best_gap:
        best_gap = gap
        best_gamma = gamma

    if mean_km_share < TARGET_TELEWORK_KM_SHARE:
        low_gamma = gamma
    else:
        high_gamma = gamma

# Final draw with calibrated exponent
telework_idx, share_km = draw_teleworkers(best_gamma, seed=int(rng_telework.integers(1_000_000_000)))

# Tag commuters
df_commuters["telework_today"] = False
df_commuters.loc[telework_idx, "telework_today"] = True

# Quick checks
share_agents = df_commuters["telework_today"].mean()
km_total = df_commuters["commute_dist_km"].sum()
km_tele = df_commuters.loc[df_commuters["telework_today"], "commute_dist_km"].sum()
share_km = km_tele / km_total if km_total > 0 else np.nan

print(f"Calibrated distance gamma: {best_gamma:.3f}")
print(f"Telework tagged agents: {n_telework} / {n_agents} ({share_agents*100:.2f}%)")
print(f"Share of commute km represented by tagged teleworkers: {share_km*100:.2f}% (target: {TARGET_TELEWORK_KM_SHARE*100:.1f}%)")
print("Telework share by education (%):")
print((df_commuters.groupby("education")["telework_today"].mean() * 100).round(2).sort_values(ascending=False))

In [ ]:
# Save to files with and without teleworkers (for separate analysis)
df_commuters.to_csv("output/commuters_with_telework_tag.csv", index=False)

df_commuters_non_tele = df_commuters[~df_commuters["telework_today"]].copy()
df_commuters_non_tele.to_csv("output/commuters_travelling.csv", index=False)

# Quick Validation

In [ ]:
import matplotlib.pyplot as plt

# Keep only rows where both distances are available
df_compare = df_commuters[["commute_dist_km", "distance_km"]].dropna().copy()

# Use a capped x-range for a readable histogram (ignore extreme outliers in axis scaling)
x_max = np.nanpercentile(
    np.r_[df_compare["commute_dist_km"].values, df_compare["distance_km"].values], 99
)
bins = np.linspace(0, x_max, 50)

plt.figure(figsize=(10, 5))
plt.hist(
    df_compare["distance_km"],
    bins=bins,
    alpha=0.5,
    density=True,
    label="Observed (distance_km)"
)
plt.hist(
    df_compare["commute_dist_km"],
    bins=bins,
    alpha=0.5,
    density=True,
    label="Estimated (commute_dist_km)"
)

plt.xlabel("Commute distance (km)")
plt.ylabel("Density")
plt.title("Distribution comparison: observed vs estimated commute distances")
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print(df_compare.describe().T[["mean", "std", "min", "50%", "max"]])